In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from math import radians, sin, cos, sqrt, atan2

from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD


# ============================================================
# 1. CONFIG
# ============================================================

DATA_PATH = "D:\HK9\HHTRQD\YelpDataset"
OUTPUT_PATH = "D:\HK9\HHTRQD\Output"

TARGET_CITY = "Philadelphia"
TARGET_STATE = "PA"

# CF
N_FACTORS = 50
RANDOM_STATE = 42

# Hybrid default weights
DEFAULT_WEIGHTS = {
    "cf": 0.4,
    "geo": 0.3,
    "cat": 0.3
}

# Geo
DEFAULT_DECAY_RATE = 0.15

# Recommendation
DEFAULT_MAX_DISTANCE_KM = 15
DEFAULT_TOP_K = 10

<>:17: SyntaxWarning: invalid escape sequence '\H'
<>:18: SyntaxWarning: invalid escape sequence '\H'
<>:17: SyntaxWarning: invalid escape sequence '\H'
<>:18: SyntaxWarning: invalid escape sequence '\H'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_30556\3994703083.py:17: SyntaxWarning: invalid escape sequence '\H'
  DATA_PATH = "D:\HK9\HHTRQD\YelpDataset"
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_30556\3994703083.py:18: SyntaxWarning: invalid escape sequence '\H'
  OUTPUT_PATH = "D:\HK9\HHTRQD\Output"


In [2]:
# ============================================================
# 2. DATA PATH
# ============================================================

business_path = os.path.join(
    DATA_PATH,
    "yelp_academic_dataset_business.json"
)

review_path = os.path.join(
    DATA_PATH,
    "yelp_academic_dataset_review.json"
)

user_path = os.path.join(
    DATA_PATH,
    "yelp_academic_dataset_user.json"
)

print("Files:")
print(os.listdir(DATA_PATH))

Files:
['Dataset_User_Agreement.pdf', 'yelp_academic_dataset_business.json', 'yelp_academic_dataset_business_Philadelphia.json', 'yelp_academic_dataset_checkin.json', 'yelp_academic_dataset_review.json', 'yelp_academic_dataset_tip.json', 'yelp_academic_dataset_user.json']


In [3]:
# ============================================================
# 3. LOAD BUSINESS DATA
# ============================================================

print("\nLoading business data...")

business_df = pd.read_json(
    business_path,
    lines=True
)

print("Business shape:", business_df.shape)


Loading business data...
Business shape: (150346, 14)


In [4]:
# ============================================================
# 4. FILTER BUSINESS BY CITY
# ============================================================

biz_city = business_df[
    (business_df["city"] == TARGET_CITY) &
    (business_df["state"] == TARGET_STATE)
].copy()

biz_city = biz_city.reset_index(drop=True)

print(
    f"\nTarget city: {TARGET_CITY}, {TARGET_STATE}"
)

print(
    "Number of businesses:",
    len(biz_city)
)

biz_ids = set(
    biz_city["business_id"]
)


Target city: Philadelphia, PA
Number of businesses: 14567


In [5]:
# ============================================================
# 5. LOAD & FILTER REVIEWS
#    Chunk processing because review.json is large
# ============================================================

print("\nFiltering reviews...")

chunksize = 200_000

review_chunks = []

for chunk in pd.read_json(
    review_path,
    lines=True,
    chunksize=chunksize
):

    filtered = chunk[
        chunk["business_id"].isin(biz_ids)
    ]

    if len(filtered) > 0:

        review_chunks.append(
            filtered[
                [
                    "review_id",
                    "user_id",
                    "business_id",
                    "stars",
                    "date"
                ]
            ]
        )


reviews_city = pd.concat(
    review_chunks,
    ignore_index=True
)

reviews_city["date"] = pd.to_datetime(
    reviews_city["date"]
)

print(
    "Reviews in target city:",
    reviews_city.shape
)


Filtering reviews...
Reviews in target city: (967517, 5)


In [6]:
# ============================================================
# 6. FILTER USERS
# ============================================================

print("\nFiltering users...")

review_user_ids = set(
    reviews_city["user_id"]
)

user_chunks = []

for chunk in pd.read_json(
    user_path,
    lines=True,
    chunksize=chunksize
):

    filtered = chunk[
        chunk["user_id"].isin(review_user_ids)
    ]

    if len(filtered) > 0:

        user_chunks.append(
            filtered[
                [
                    "user_id",
                    "name",
                    "review_count",
                    "yelping_since"
                ]
            ]
        )


users_city = pd.concat(
    user_chunks,
    ignore_index=True
)

print(
    "Users in target city:",
    users_city.shape
)


Filtering users...
Users in target city: (279847, 4)


In [7]:
# ============================================================
# 7. SAVE RAW FILTERED DATA
# ============================================================

biz_city.to_parquet(
    os.path.join(
        OUTPUT_PATH,
        "business_city.parquet"
    ),
    index=False
)

reviews_city.to_parquet(
    os.path.join(
        OUTPUT_PATH,
        "reviews_city.parquet"
    ),
    index=False
)

users_city.to_parquet(
    os.path.join(
        OUTPUT_PATH,
        "users_city.parquet"
    ),
    index=False
)

print("\nFiltered data saved.")


Filtered data saved.


In [9]:
# ============================================================
# 8. TIME-BASED TRAIN / TEST SPLIT
# ============================================================

reviews_city = reviews_city.sort_values(
    "date"
).reset_index(drop=True)

split_idx = int(
    len(reviews_city) * 0.8
)

split_date = reviews_city.loc[
    split_idx,
    "date"
]

train_reviews = reviews_city[
    reviews_city["date"] <= split_date
].copy()

test_reviews = reviews_city[
    reviews_city["date"] > split_date
].copy()

print("\nTime split:")
print("Split date:", split_date)
print("Train:", train_reviews.shape)
print("Test :", test_reviews.shape)


# Save train/test data
train_reviews.to_parquet(
    os.path.join(
        OUTPUT_PATH,
        "train_reviews.parquet"
    ),
    index=False
)

test_reviews.to_parquet(
    os.path.join(
        OUTPUT_PATH,
        "test_reviews.parquet"
    ),
    index=False
)


Time split:
Split date: 2019-02-17 22:54:32
Train: (774014, 5)
Test : (193503, 5)


In [10]:
# ============================================================
# 9. IMPORTANT:
#    AGGREGATE DUPLICATE USER-BUSINESS INTERACTIONS
# ============================================================

# Một user có thể review cùng business nhiều lần.
# Không đưa trực tiếp các duplicate rows vào sparse matrix
# vì csr_matrix có thể cộng các giá trị trùng (row, col).

train_interactions = (
    train_reviews
    .groupby(
        ["user_id", "business_id"],
        as_index=False
    )
    .agg(
        stars=("stars", "mean")
    )
)

print(
    "\nUnique train interactions:",
    train_interactions.shape
)


Unique train interactions: (750088, 3)


In [11]:
# ============================================================
# 10. BUILD USER-ITEM MATRIX
# ============================================================

user_ids_list = (
    train_interactions["user_id"]
    .unique()
)

biz_ids_list = (
    biz_city["business_id"]
    .unique()
)

user_idx = {
    user_id: idx
    for idx, user_id
    in enumerate(user_ids_list)
}

biz_idx = {
    business_id: idx
    for idx, business_id
    in enumerate(biz_ids_list)
}


rows = train_interactions[
    "user_id"
].map(user_idx)

cols = train_interactions[
    "business_id"
].map(biz_idx)

data = train_interactions[
    "stars"
].values


# Safety mask
mask = (
    rows.notna() &
    cols.notna()
)

rows = rows[mask].astype(int)
cols = cols[mask].astype(int)
data = data[mask]


interaction_matrix = csr_matrix(
    (
        data,
        (rows, cols)
    ),
    shape=(
        len(user_ids_list),
        len(biz_ids_list)
    )
)

print(
    "\nInteraction matrix:",
    interaction_matrix.shape
)

print(
    "Non-zero interactions:",
    interaction_matrix.nnz
)


Interaction matrix: (218030, 14567)
Non-zero interactions: 750088


In [12]:
# ============================================================
# 11. TRAIN SVD MODEL
# ============================================================

svd = TruncatedSVD(
    n_components=N_FACTORS,
    random_state=RANDOM_STATE
)

user_factors = svd.fit_transform(
    interaction_matrix
)

item_factors = svd.components_.T

print(
    "\nSVD trained."
)

print(
    "User factors:",
    user_factors.shape
)

print(
    "Item factors:",
    item_factors.shape
)


SVD trained.
User factors: (218030, 50)
Item factors: (14567, 50)


In [13]:
# ============================================================
# 12. TRAIN BUSINESS RATING
#     Used ONLY for CF cold-start fallback
# ============================================================

train_business_rating = (
    train_reviews
    .groupby("business_id")["stars"]
    .mean()
    .to_dict()
)

GLOBAL_TRAIN_MEAN = float(
    train_reviews["stars"].mean()
)

print(
    "\nGlobal train rating:",
    round(GLOBAL_TRAIN_MEAN, 3)
)


Global train rating: 3.776


In [14]:
# ============================================================
# 13. CF SCORE
# ============================================================

def get_cf_score(
    user_id,
    business_id
):
    """
    Return CF score normalized to [0, 1].

    Normal case:
        SVD user factor dot item factor.

    Cold-start:
        Use business average rating calculated
        ONLY from training data.
    """

    if (
        user_id in user_idx
        and business_id in biz_idx
    ):

        raw = (
            user_factors[
                user_idx[user_id]
            ]
            @
            item_factors[
                biz_idx[business_id]
            ]
        )

        return float(
            np.clip(
                raw / 5.0,
                0.0,
                1.0
            )
        )

    # Cold-start fallback
    avg_rating = train_business_rating.get(
        business_id,
        GLOBAL_TRAIN_MEAN
    )

    return float(
        np.clip(
            avg_rating / 5.0,
            0.0,
            1.0
        )
    )

In [15]:
# ============================================================
# 14. GEO SCORE
# ============================================================

def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):
    """
    Calculate distance between two coordinates.
    """

    R = 6371.0

    dlat = radians(
        lat2 - lat1
    )

    dlon = radians(
        lon2 - lon1
    )

    a = (
        sin(dlat / 2) ** 2
        +
        cos(radians(lat1))
        *
        cos(radians(lat2))
        *
        sin(dlon / 2) ** 2
    )

    return (
        R
        *
        2
        *
        atan2(
            sqrt(a),
            sqrt(1 - a)
        )
    )


def haversine_vectorized(
    user_lat,
    user_lon,
    latitudes,
    longitudes
):
    """
    Vectorized Haversine calculation.
    Faster than DataFrame.apply().
    """

    R = 6371.0

    lat1 = np.radians(user_lat)
    lon1 = np.radians(user_lon)

    lat2 = np.radians(latitudes)
    lon2 = np.radians(longitudes)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        *
        np.cos(lat2)
        *
        np.sin(dlon / 2) ** 2
    )

    return (
        R
        *
        2
        *
        np.arctan2(
            np.sqrt(a),
            np.sqrt(1 - a)
        )
    )


def get_geo_score(
    user_lat,
    user_lon,
    biz_lat,
    biz_lon,
    decay_rate=DEFAULT_DECAY_RATE
):
    """
    Exponential distance decay.

    score = exp(-decay_rate * distance_km)

    Near POI -> score close to 1
    Far POI  -> score approaches 0
    """

    distance = haversine_km(
        user_lat,
        user_lon,
        biz_lat,
        biz_lon
    )

    return float(
        np.exp(
            -decay_rate * distance
        )
    )

In [16]:
# ============================================================
# 15. CATEGORY SCORE
# ============================================================

def parse_categories(
    cat_string
):
    """
    Convert Yelp category string into set.
    """

    if pd.isna(cat_string):
        return set()

    return set(
        c.strip()
        for c in cat_string.split(",")
        if c.strip()
    )


biz_city["category_set"] = (
    biz_city["categories"]
    .apply(parse_categories)
)


def get_category_score(
    query_categories,
    biz_categories
):
    """
    Jaccard similarity:

        |query ∩ business|
        -----------------
        |query ∪ business|

    Result in [0, 1].
    """

    if (
        not query_categories
        or not biz_categories
    ):
        return 0.0

    intersection = len(
        query_categories
        &
        biz_categories
    )

    union = len(
        query_categories
        |
        biz_categories
    )

    if union == 0:
        return 0.0

    return (
        intersection / union
    )

In [17]:
# ============================================================
# 16. NORMALIZE / VALIDATE WEIGHTS
# ============================================================

def normalize_weights(
    weights=None
):
    """
    Validate and normalize hybrid weights.

    Example:
        {
            'cf': 0.4,
            'geo': 0.3,
            'cat': 0.3
        }

    The returned weights always sum to 1.
    """

    if weights is None:
        weights = DEFAULT_WEIGHTS.copy()

    required_keys = {
        "cf",
        "geo",
        "cat"
    }

    if set(weights.keys()) != required_keys:
        raise ValueError(
            "Weights must contain exactly: "
            "'cf', 'geo', 'cat'"
        )

    if any(
        value < 0
        for value in weights.values()
    ):
        raise ValueError(
            "Weights must be non-negative."
        )

    total = sum(
        weights.values()
    )

    if total <= 0:
        raise ValueError(
            "Sum of weights must be greater than 0."
        )

    return {
        key: value / total
        for key, value in weights.items()
    }

In [18]:
# ============================================================
# 17. HYBRID SCORE + EXPLAINABILITY
# ============================================================

def hybrid_score(
    cf,
    geo,
    cat,
    weights=None
):
    """
    Weighted Sum Model.

    Returns:
        total_score
        breakdown
    """

    w = normalize_weights(
        weights
    )

    contrib_cf = (
        w["cf"] * cf
    )

    contrib_geo = (
        w["geo"] * geo
    )

    contrib_cat = (
        w["cat"] * cat
    )

    total = (
        contrib_cf
        +
        contrib_geo
        +
        contrib_cat
    )

    if total > 0:

        cf_pct = (
            contrib_cf
            /
            total
            *
            100
        )

        geo_pct = (
            contrib_geo
            /
            total
            *
            100
        )

        cat_pct = (
            contrib_cat
            /
            total
            *
            100
        )

    else:

        cf_pct = 0
        geo_pct = 0
        cat_pct = 0


    breakdown = {

        # Raw scores
        "cf_score": round(
            cf,
            4
        ),

        "geo_score": round(
            geo,
            4
        ),

        "cat_score": round(
            cat,
            4
        ),

        # Weights
        "cf_weight": round(
            w["cf"],
            4
        ),

        "geo_weight": round(
            w["geo"],
            4
        ),

        "cat_weight": round(
            w["cat"],
            4
        ),

        # Contribution
        "cf_contribution": round(
            contrib_cf,
            4
        ),

        "geo_contribution": round(
            contrib_geo,
            4
        ),

        "cat_contribution": round(
            contrib_cat,
            4
        ),

        # Contribution percentage
        "cf_contrib_pct": round(
            cf_pct,
            2
        ),

        "geo_contrib_pct": round(
            geo_pct,
            2
        ),

        "cat_contrib_pct": round(
            cat_pct,
            2
        )
    }

    return (
        float(total),
        breakdown
    )

In [19]:
# ============================================================
# 18. RECOMMEND POIs
# ============================================================

def recommend_pois(
    user_lat,
    user_lon,
    query_categories,
    user_id=None,
    top_k=DEFAULT_TOP_K,
    weights=None,
    max_distance_km=DEFAULT_MAX_DISTANCE_KM,
    decay_rate=DEFAULT_DECAY_RATE
):
    """
    Main scoring function.

    Input:
        user_lat
        user_lon
        query_categories
        user_id
        top_k
        weights
        max_distance_km

    Output:
        DataFrame ranked by hybrid score.
    """

    # --------------------------------------------------------
    # 1. Copy candidate POIs
    # --------------------------------------------------------

    candidates = biz_city.copy()

    # --------------------------------------------------------
    # 2. Calculate distance vectorized
    # --------------------------------------------------------

    candidates["dist_km"] = (
        haversine_vectorized(
            user_lat,
            user_lon,
            candidates["latitude"].values,
            candidates["longitude"].values
        )
    )

    # --------------------------------------------------------
    # 3. Filter by maximum distance
    # --------------------------------------------------------

    candidates = candidates[
        candidates["dist_km"]
        <= max_distance_km
    ].copy()

    # --------------------------------------------------------
    # 4. Calculate scores
    # --------------------------------------------------------

    results = []

    for _, row in candidates.iterrows():

        # CF
        if user_id is not None:

            cf = get_cf_score(
                user_id,
                row["business_id"]
            )

        else:

            # Anonymous user:
            # use train business rating as baseline
            cf = train_business_rating.get(
                row["business_id"],
                GLOBAL_TRAIN_MEAN
            ) / 5.0

            cf = float(
                np.clip(
                    cf,
                    0.0,
                    1.0
                )
            )

        # Geo
        geo = get_geo_score(
            user_lat,
            user_lon,
            row["latitude"],
            row["longitude"],
            decay_rate=decay_rate
        )

        # Category
        cat = get_category_score(
            query_categories,
            row["category_set"]
        )

        # Hybrid
        score, breakdown = hybrid_score(
            cf,
            geo,
            cat,
            weights
        )

        results.append({

            "business_id":
                row["business_id"],

            "name":
                row["name"],

            "latitude":
                row["latitude"],

            "longitude":
                row["longitude"],

            "categories":
                row["categories"],

            "stars":
                row["stars"],

            "dist_km":
                round(
                    row["dist_km"],
                    2
                ),

            "score":
                round(
                    score,
                    4
                ),

            **breakdown
        })


    if not results:

        return pd.DataFrame()


    result_df = pd.DataFrame(
        results
    )

    # --------------------------------------------------------
    # 5. Ranking
    # --------------------------------------------------------

    result_df = (
        result_df
        .sort_values(
            "score",
            ascending=False
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    return result_df

In [20]:
# ============================================================
# 19. SAMPLE RECOMMENDATION
# ============================================================

# Dùng trung bình tọa độ của city làm demo location.
# Sau này frontend sẽ truyền location thật.

sample_lat = biz_city[
    "latitude"
].mean()

sample_lon = biz_city[
    "longitude"
].mean()


sample_result = recommend_pois(

    user_lat=sample_lat,

    user_lon=sample_lon,

    query_categories={
        "Coffee & Tea"
    },

    user_id=None,

    top_k=10,

    weights={
        "cf": 0.3,
        "geo": 0.4,
        "cat": 0.3
    },

    max_distance_km=15
)


print("\nSample recommendation:")
print(
    sample_result[
        [
            "business_id",
            "name",
            "stars",
            "dist_km",
            "score",
            "cf_score",
            "geo_score",
            "cat_score",
            "cf_contrib_pct",
            "geo_contrib_pct",
            "cat_contrib_pct"
        ]
    ]
)


Sample recommendation:
              business_id                    name  stars  dist_km   score  \
0  5asvaAT-jDbOnCM3iPlScw       ReAnimator Coffee    4.5     1.09  0.7451   
1  4t3yeXmkrMaJruCWA0H8DA       Saxbys Temple Fox    4.0     0.53  0.7443   
2  LzgxfMu24HbJbUi1cCk70Q           Premium Steap    4.5     2.31  0.7329   
3  61RiTA-T02sU84FhoCLz_g           The Lola Bean    4.5     1.68  0.7260   
4  GEtK2S6zALXgsjKjK3abJw               Starbucks    3.0     2.52  0.7243   
5  RMRlzkMedvSjxFLN9cIJSw            Coffee House    4.0     1.59  0.7159   
6  vdpxxb6fxjYtJMnW_V0GZg  Lucky Goat Coffeehouse    4.5     2.31  0.7051   
7  QqOKPUl7aNiW4IN_Ij2mBg            Café Bustelo    4.0     1.52  0.6949   
8  sD2yBIzp0SNYRMD4QfH9Jw               Starbucks    2.5     1.52  0.6948   
9  -QG6KSRQKTQ80--wqrnLTg               Starbucks    4.0     2.79  0.6933   

   cf_score  geo_score  cat_score  cf_contrib_pct  geo_contrib_pct  \
0    0.8516     0.8491        0.5           34.29         

In [21]:
# ============================================================
# 20. EXPORT sample_scores.csv
# ============================================================

sample_scores_path = os.path.join(
    OUTPUT_PATH,
    "sample_scores.csv"
)

sample_result.to_csv(
    sample_scores_path,
    index=False
)

print(
    "\nSample scores saved to:"
)

print(
    sample_scores_path
)


Sample scores saved to:
D:\HK9\HHTRQD\Output\sample_scores.csv


In [22]:
# ============================================================
# 21. EXPORT CF MODEL ARTIFACT
# ============================================================

cf_model_artifact = {

    "user_factors":
        user_factors,

    "item_factors":
        item_factors,

    "user_idx":
        user_idx,

    "biz_idx":
        biz_idx,

    "train_business_rating":
        train_business_rating,

    "global_train_mean":
        GLOBAL_TRAIN_MEAN,

    "n_factors":
        N_FACTORS,

    "random_state":
        RANDOM_STATE,

    "target_city":
        TARGET_CITY,

    "target_state":
        TARGET_STATE
}


cf_model_path = os.path.join(
    OUTPUT_PATH,
    "cf_model.pkl"
)

joblib.dump(
    cf_model_artifact,
    cf_model_path
)

print(
    "\nCF model saved to:"
)

print(
    cf_model_path
)


CF model saved to:
D:\HK9\HHTRQD\Output\cf_model.pkl


In [23]:
# ============================================================
# 22. EXPORT BUSINESS ARTIFACT
# ============================================================

business_artifact_path = os.path.join(
    OUTPUT_PATH,
    "business_city.pkl"
)

joblib.dump(
    biz_city,
    business_artifact_path
)

print(
    "Business artifact saved to:"
)

print(
    business_artifact_path
)

Business artifact saved to:
D:\HK9\HHTRQD\Output\business_city.pkl


In [24]:
# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)

print("DATA & SCORING ENGINE COMPLETED")

print("=" * 60)

print(
    f"City: {TARGET_CITY}, {TARGET_STATE}"
)

print(
    f"Businesses: {len(biz_city):,}"
)

print(
    f"Reviews: {len(reviews_city):,}"
)

print(
    f"Users: {len(users_city):,}"
)

print(
    f"Train interactions: "
    f"{len(train_interactions):,}"
)

print(
    f"Train/Test split: "
    f"{len(train_reviews):,} / "
    f"{len(test_reviews):,}"
)

print("\nArtifacts:")

print(
    "- business_city.parquet"
)

print(
    "- reviews_city.parquet"
)

print(
    "- users_city.parquet"
)

print(
    "- train_reviews.parquet"
)

print(
    "- test_reviews.parquet"
)

print(
    "- cf_model.pkl"
)

print(
    "- business_city.pkl"
)

print(
    "- sample_scores.csv"
)

print("=" * 60)


DATA & SCORING ENGINE COMPLETED
City: Philadelphia, PA
Businesses: 14,567
Reviews: 967,517
Users: 279,847
Train interactions: 750,088
Train/Test split: 774,014 / 193,503

Artifacts:
- business_city.parquet
- reviews_city.parquet
- users_city.parquet
- train_reviews.parquet
- test_reviews.parquet
- cf_model.pkl
- business_city.pkl
- sample_scores.csv
